In [10]:
    import os
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    import tensorflow as tf
    from tensorflow import keras



    import joblib

# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_csv("synthetic_health_lifestyle_dataset.csv")
df.head()


,ID,Age,Gender,Height_cm,Weight_kg,BMI,Smoker,Exercise_Freq,Diet_Quality,Alcohol_Consumption,Chronic_Disease,Stress_Level,Sleep_Hours
0,1,56,Other,177.6,37.3,11.8,Yes,NaN,Poor,NaN,No,9,8.5
1,2,69,Other,169.3,70.7,24.7,No,1-2 times/week,Good,High,No,2,5.9
2,3,46,Female,159.1,69.0,27.3,No,Daily,Excellent,Moderate,No,3,4.8
3,4,32,Male,170.6,76.4,26.3,No,3-5 times/week,Excellent,Moderate,No,9,6.6
4,5,60,Male,158.4,60.4,24.1,No,3-5 times/week,Excellent,Low,Yes,6,6.1


In [11]:
def compute_risk(row):
    score = 0.0

    # BMI
    bmi = row["BMI"]
    if bmi < 18.5:
        score += 0.5
    elif bmi < 25:
        score += 0
    elif bmi < 30:
        score += 1
    else:
        score += 2

    # Smoking
    if row["Smoker"] == "Yes":
        score += 2

    # Exercise
    ex = row["Exercise_Freq"]
    if ex == "None":
        score += 2
    elif ex == "1-2 times/week":
        score += 1
    elif ex == "Daily":
        score -= 0.5

    # Diet
    diet = row["Diet_Quality"]
    if diet == "Poor":
        score += 2
    elif diet == "Excellent":
        score -= 0.5

    # Alcohol
    alc = row["Alcohol_Consumption"]
    if alc == "Low":
        score += 0.5
    elif alc == "Moderate":
        score += 1
    elif alc == "High":
        score += 2

    # Chronic disease
    if row["Chronic_Disease"] == "Yes":
        score += 3

    # Stress
    score += (row["Stress_Level"] - 5) * 0.3

    # Sleep
    h = row["Sleep_Hours"]
    if h < 6:
        score += 2
    elif h < 7:
        score += 1
    elif h > 9:
        score += 0.5

    return score

df["Risk_Score"] = df.apply(compute_risk, axis=1)

# Cut into 3 classes
df["Risk_Label"] = pd.cut(
    df["Risk_Score"],
    bins=[-10, 3.5, 7.0, 30],
    labels=[0, 1, 2]     # 0=Low, 1=Moderate, 2=High
).astype(int)

df["Risk_Label"].value_counts()


Risk_Label
1    3505
0    2942
2    1053
Name: count, dtype: int64

In [12]:
# Categorical columns
cat_cols = [
    "Gender", "Smoker", "Exercise_Freq",
    "Diet_Quality", "Alcohol_Consumption", "Chronic_Disease"
]

df_ml = df.copy()
df_ml = pd.get_dummies(df_ml, columns=cat_cols, drop_first=True)

# Features & label
feature_cols = [c for c in df_ml.columns if c not in ["ID", "Risk_Score", "Risk_Label"]]
X = df_ml[feature_cols].values.astype("float32")
y = df_ml["Risk_Label"].values.astype("int32")

X.shape, y.shape


((7500, 17), (7500,))

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save scaler + feature cols for later use in Streamlit
os.makedirs("src/models", exist_ok=True)
joblib.dump(
    {"scaler": scaler, "feature_cols": feature_cols},
    "src/models/tf_metadata.pkl"
)
print("Saved metadata to src/models/tf_metadata.pkl")


Saved metadata to src/models/tf_metadata.pkl


In [24]:

num_classes = 3

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
])


model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1,
)

loss, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print("Test accuracy:", acc)


Epoch 1/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4472 - loss: 1.0316 - val_accuracy: 0.7458 - val_loss: 0.5755
Epoch 2/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7749 - loss: 0.5196 - val_accuracy: 0.7933 - val_loss: 0.4582
Epoch 3/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7986 - loss: 0.4440 - val_accuracy: 0.7992 - val_loss: 0.4469
Epoch 4/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7991 - loss: 0.4369 - val_accuracy: 0.8017 - val_loss: 0.4501
Epoch 5/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8134 - loss: 0.4081 - val_accuracy: 0.7983 - val_loss: 0.4481
Epoch 6/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8092 - loss: 0.4214 - val_accuracy: 0.7975 - val_loss: 0.4472
Epoch 7/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8197 - loss: 0.3982 - val_accuracy: 0.8158 - val_loss: 0.4364
Epoch 8/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8264 - loss: 0.3945 - val_accuracy: 0.

In [25]:
model.save("src/models/health_risk_tf_model.h5")
print("Saved TF model.")


Saved TF model.


In [26]:
import joblib

joblib.dump(
    {"scaler": scaler, "feature_cols": feature_cols},
    "src/models/tf_metadata.pkl"
)

print("Saved metadata.")


Saved metadata.
